<a href="https://colab.research.google.com/github/kkl5524/oasis-plus/blob/main/synthea_data_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# java
!apt-get update
!apt-get install -y openjdk-11-jdk-headless

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

!java -version

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.4 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,153 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main 

In [5]:
# libraries and frameworks (not java)

import json

In [2]:
# actual import of synthea

!git clone https://github.com/synthetichealth/synthea.git

Cloning into 'synthea'...
remote: Enumerating objects: 73471, done.
remote: Counting objects: 100% (1109/1109), done.
remote: Compressing objects: 100% (249/249), done.
remote: Total 73471 (delta 985), reused 864 (delta 859), pack-reused 72362 (from 3)
Receiving objects: 100% (73471/73471), 760.28 MiB | 23.79 MiB/s, done.
Resolving deltas: 100% (43769/43769), done.
/content/synthea
.............10%.............20%.............30%.............40%.............50%.............60%.............70%.............80%.............90%..............100%

Welcome to Gradle 8.14.3!

Here are the highlights of this release:
 - Java 24 support
 - GraalVM Native Image toolchain selection
 - Enhancements to test reporting
 - Build Authoring improvements

For more details see https://docs.gradle.org/8.14.3/release-notes.html

Starting a Gradle Daemon (subsequent builds will be faster)


> Starting Daemon> IDLE<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZING s]<-------------> 0% INITIALIZ

In [3]:
# configurations for synthea (mimic-iii settings)

config_additions = """

exporter.mimic.export = true
exporter.mimic.folder = output/mimic
exporter.csv.export = true
exporter.baseDirectory = ./output/
exporter.use_uuid_filenames = false
exporter.subfolders_by_id_substring = false

# Generate more realistic ICU scenarios
generate.icustay.prob = 0.20
generate.icustay.mean_los_days = 3.5
generate.icustay.sd_los_days   = 2.0

generate.vitals.frequency = 5
generate.labs.frequency = 24   # 24 per day

generate.notes.probability = 1.0
generate.notes.per.encounter = 5

generate.demographics.socioeconomic.income.poverty = 0.18
generate.demographics.socioeconomic.education.less_than_hs.percentage = 0.10

exporter.years_of_history = 10

"""

with open('src/main/resources/synthea.properties', 'a') as f:
    f.write(config_additions)

print("Configuration updated")

Configuration updated


In [ ]:
MODULE_DIR = "synthea/src/main/resources/modules"

In [ ]:
# helper functions
def modify_json(path, callback):
    """Load → modify → save JSON"""
    with open(path, "r") as f:
        data = json.load(f)

    modified = callback(data)

    with open(path, "w") as f:
        json.dump(modified, f, indent=2)

def modify_icu_module(data):
    for state in data["states"].values():
        # longer ICU stays (MIMIC-style)
        if "distribution" in state and "mean" in state["distribution"]:
            state["distribution"]["mean"] = max(state["distribution"]["mean"], 3.5)
            state["distribution"]["minimum"] = 1.0

        # add higher chance of ventilation
        if "direct_transition" in state and state["direct_transition"] == "Ventilation":
            state["transition_probability"] = 0.35  # increase vent probability to MIMIC-like levels

    return data

def modify_vitals_module(data):
    for state in data["states"].values():
        if state.get("type") == "VitalSign":
            state["frequency"] = {"quantity": 15, "unit": "minutes"}  # MIMIC-style frequency
    return data

In [ ]:
icu_path = os.path.join(MODULE_DIR, "hospital", "icu.json")
modify_json(icu_path, modify_icu_module)

vitals_path = os.path.join(MODULE_DIR, "observation", "vital_signs.json")
modify_json(vitals_path, modify_vitals_module)

In [ ]:
%cd synthea
!./gradlew build check test

In [ ]:
# Generate 5000 patients
!./run_synthea -s 123 -p 3000 Massachusetts

print("\nPatient generation complete")



<-------------> 0% INITIALIZING s]> Evaluating settings<-------------> 0% CONFIGURING s]> root project<-------------> 0% CONFIGURING s]> root project > Resolve files of configuration 'classpath'<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]> root project<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]<-------------> 0% CONFIGURING s]<==-----------> 20% EXECUTING [1s]> :compileJava > Resolve dependencies of :compileClasspath> :compileJava<=====--------> 40% EXECUTING [1s]> :processResources<=====--------> 40% EXECUTING [2s]<==========---> 80% EXECUTING [2s]> :run > Resolve dependencies of :runtimeClasspath> :run<==========---> 80% EXECUTING [3s]<==========---> 80% EXECUTING [4s]<==========---> 80% EXECUTING [5s]<==========---> 80% EXECUTING [6s]<==========---> 80% EXECUTING [7s]<==========---> 80% EXECUTING [8s]<==========---> 80% EXECUTING [9s]<==========---> 80% EXECUTING [10s]<==========---> 80% EXECUTING [11s]


In [88]:
# Check output
!ls -lh output/csv/

total 4.0G
-rw-r--r-- 1 root root 896K Nov 17 03:22 allergies.csv
-rw-r--r-- 1 root root 3.7M Nov 17 03:22 careplans.csv
-rw-r--r-- 1 root root 228M Nov 17 03:22 claims.csv
-rw-r--r-- 1 root root 2.5G Nov 17 03:22 claims_transactions.csv
-rw-r--r-- 1 root root  32M Nov 17 03:22 conditions.csv
-rw-r--r-- 1 root root 7.2M Nov 17 03:22 devices.csv
-rw-r--r-- 1 root root 105M Nov 17 03:22 encounters.csv
-rw-r--r-- 1 root root 169M Nov 17 03:22 imaging_studies.csv
-rw-r--r-- 1 root root  12M Nov 17 03:22 immunizations.csv
-rw-r--r-- 1 root root  73M Nov 17 03:22 medications.csv
-rw-r--r-- 1 root root 739M Nov 17 03:22 observations.csv
-rw-r--r-- 1 root root 165K Nov 17 03:22 organizations.csv
-rw-r--r-- 1 root root 1.6M Nov 17 03:22 patients.csv
-rw-r--r-- 1 root root 1.8K Nov 17 03:22 payers.csv
-rw-r--r-- 1 root root  32M Nov 17 03:22 payer_transitions.csv
-rw-r--r-- 1 root root 198M Nov 17 03:22 procedures.csv
-rw-r--r-- 1 root root 192K Nov 17 03:22 providers.csv
-rw-r--r-- 1 root root 

In [ ]:
import zipfile

zip_path = "synthea_mimic_output.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk("synthea/output/mimic"):
        for file in files:
            full_path = os.path.join(root, file)
            zipf.write(full_path)


In [ ]:
from google.colab import files
files.download(zip_path)

### work in progress

In [89]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Loading Synthea output")

# Load all Synthea CSVs
patients_raw = pd.read_csv('output/csv/patients.csv')
encounters_raw = pd.read_csv('output/csv/encounters.csv')
conditions_raw = pd.read_csv('output/csv/conditions.csv')
procedures_raw = pd.read_csv('output/csv/procedures.csv')
medications_raw = pd.read_csv('output/csv/medications.csv')
observations_raw = pd.read_csv('output/csv/observations.csv')

print(f"Loaded data:")
print(f"  - {len(patients_raw)} patients")
print(f"  - {len(encounters_raw)} encounters")
print(f"  - {len(conditions_raw)} conditions")
print(f"  - {len(procedures_raw)} procedures")
print(f"  - {len(observations_raw)} observations")

Loading Synthea output
Loaded data:
  - 5732 patients
  - 330127 encounters
  - 208055 conditions
  - 927091 procedures
  - 4372366 observations


In [ ]:
# Transform to MIMIC-III PATIENTS table

print("CREATING TABLE 1/12: PATIENTS")

subject_id_start = 10001
subject_id_map = dict(zip(
    patients_raw['Id'],
    range(subject_id_start, subject_id_start + len(patients_raw))
))

PATIENTS = pd.DataFrame({
    'ROW_ID': range(1, len(patients_raw) + 1),
    'SUBJECT_ID': patients_raw['Id'].map(subject_id_map),
    'GENDER': patients_raw['GENDER'],
    'DOB': pd.to_datetime(patients_raw['BIRTHDATE']),
    'DOD': pd.to_datetime(patients_raw['DEATHDATE']),
    'DOD_HOSP': pd.to_datetime(patients_raw['DEATHDATE']),
    'DOD_SSN': pd.to_datetime(patients_raw['DEATHDATE']),
    'EXPIRE_FLAG': patients_raw['DEATHDATE'].notna().astype(int)
})

deathdate_map = dict(zip(
    patients_raw['Id'],
    patients_raw['DEATHDATE']
))

all_deathdates = list(deathdate_map.values())
deathdates_series = pd.Series(all_deathdates)
deathdates_series = deathdates_series.dropna()
unique_deathdates = deathdates_series.unique()
print(unique_deathdates)

print(f"PATIENTS table created: {len(PATIENTS)} rows")
print(PATIENTS.head())

print("MIMIC-III PATIENTS Table:")
print(f"\nShape: {PATIENTS.shape}")


CREATING TABLE 1/12: PATIENTS
['2007-03-29' '1995-06-24' '1944-08-14' '1998-04-06' '2004-02-23'
 '2023-12-20' '1994-10-06' '1964-12-02' '1986-06-06' '2019-07-06'
 '2007-02-02' '1993-11-11' '2013-10-12' '1971-06-20' '2003-06-28'
 '2022-11-09' '2017-12-10']
PATIENTS table created: 117 rows
   ROW_ID  SUBJECT_ID GENDER        DOB DOD DOD_HOSP DOD_SSN  EXPIRE_FLAG
0       1       10001      M 1972-05-18 NaT      NaT     NaT            0
1       2       10002      F 1978-02-03 NaT      NaT     NaT            0
2       3       10003      F 2016-04-11 NaT      NaT     NaT            0
3       4       10004      M 1978-06-04 NaT      NaT     NaT            0
4       5       10005      M 1986-01-04 NaT      NaT     NaT            0
MIMIC-III PATIENTS Table:

Shape: (117, 8)


In [ ]:
print("CREATING TABLE 2/12: ADMISSIONS")

inpatient = encounters_raw[
    encounters_raw['ENCOUNTERCLASS'].isin(['inpatient', 'emergency', 'urgent'])
].copy()

admit = pd.to_datetime(inpatient['START']).dt.tz_localize(None)
disch = pd.to_datetime(inpatient['STOP']).dt.tz_localize(None)

patient_death = inpatient['PATIENT'].map(lambda x: deathdate_map.get(x))
patient_death = pd.to_datetime(patient_death).dt.tz_localize(None)

hospital_expire_flag = ((patient_death >= admit) & (patient_death <= disch)).astype(int)
death_time = patient_death.where(hospital_expire_flag == 1)

# Create HADM_ID mapping
hadm_id_start = 100001
hadm_id_map = dict(zip(
    inpatient['Id'],
    range(hadm_id_start, hadm_id_start + len(inpatient))
))

ADMISSIONS = pd.DataFrame({
    'ROW_ID': range(1, len(inpatient) + 1),
    'SUBJECT_ID': inpatient['PATIENT'].map(subject_id_map),
    'HADM_ID': inpatient['Id'].map(hadm_id_map),
    'ADMITTIME': admit,
    'DISCHTIME': disch,
    'DEATHTIME': death_time,
    'ADMISSION_TYPE': inpatient['ENCOUNTERCLASS'].str.upper(),
    'ADMISSION_LOCATION': 'EMERGENCY ROOM ADMIT',
    'DISCHARGE_LOCATION': ['DEAD/EXPIRED' if flag == 1 else 'HOME' for flag in hospital_expire_flag],
    'INSURANCE': 'Medicare',
    'LANGUAGE': 'ENGL',
    'MARITAL_STATUS': 'MARRIED',
    'ETHNICITY': 'WHITE',
    'DIAGNOSIS': inpatient['REASONDESCRIPTION'],
    'HOSPITAL_EXPIRE_FLAG': hospital_expire_flag,
    'HAS_CHARTEVENTS_DATA': 1
})

# Add synthetic admission IDs
ADMISSIONS['ADMISSION_ID'] = range(1, len(ADMISSIONS) + 1)

print(f"ADMISSIONS table created: {len(ADMISSIONS)} rows")
print(f"Hospital mortality rate: {ADMISSIONS['HOSPITAL_EXPIRE_FLAG'].mean():.2%}")

CREATING TABLE 2/12: ADMISSIONS
ADMISSIONS table created: 585 rows
Hospital mortality rate: 0.17%


In [ ]:
print("CREATING TABLE 3/12: ICUSTAYS")

# Assume ~40% of inpatient stays involve ICU
icu_admissions = ADMISSIONS.sample(frac=0.4, random_state=42).copy()

icustay_id_start = 200001

ICUSTAYS = pd.DataFrame({
    'ROW_ID': range(1, len(icu_admissions) + 1),
    'SUBJECT_ID': icu_admissions['SUBJECT_ID'],
    'HADM_ID': icu_admissions['HADM_ID'],
    'ICUSTAY_ID': range(icustay_id_start, icustay_id_start + len(icu_admissions)),
    'DBSOURCE': 'metavision',
    'FIRST_CAREUNIT': np.random.choice(['MICU', 'SICU', 'CCU', 'CSRU'], len(icu_admissions)),
    'LAST_CAREUNIT': np.random.choice(['MICU', 'SICU', 'CCU', 'CSRU'], len(icu_admissions)),
    'FIRST_WARDID': np.random.randint(1, 50, len(icu_admissions)),
    'LAST_WARDID': np.random.randint(1, 50, len(icu_admissions)),
    'INTIME': icu_admissions['ADMITTIME'] + pd.to_timedelta(
        np.random.randint(0, 48, len(icu_admissions)), unit='h'
    ),
    'OUTTIME': icu_admissions['DISCHTIME']
})

# Calculate LOS
ICUSTAYS['LOS'] = (ICUSTAYS['OUTTIME'] - ICUSTAYS['INTIME']).dt.total_seconds() / 86400

print(f"ICUSTAYS table created: {len(ICUSTAYS)} rows")
print(f"Mean LOS: {ICUSTAYS['LOS'].mean():.2f} days")


CREATING TABLE 3/12: ICUSTAYS
ICUSTAYS table created: 234 rows
Mean LOS: 1.51 days


In [ ]:
print("CREATING TABLE 4/12: D_ITEMS (Dictionary)")

vital_items = [
    # Vital Signs
    (211, 'Heart Rate', 'HR', 'Numeric', 'CareVue', 'Vital Signs', '/min', 'metavision'),
    (220045, 'Heart Rate', 'HR', 'Numeric', 'Metavision', 'Vital Signs', 'bpm', 'metavision'),

    (220050, 'Arterial Blood Pressure systolic', 'ABPsys', 'Numeric', 'Metavision', 'Vital Signs', 'mmHg', 'metavision'),
    (220051, 'Arterial Blood Pressure diastolic', 'ABPdias', 'Numeric', 'Metavision', 'Vital Signs', 'mmHg', 'metavision'),
    (220052, 'Arterial Blood Pressure mean', 'ABPmean', 'Numeric', 'Metavision', 'Vital Signs', 'mmHg', 'metavision'),
    (220179, 'Non Invasive Blood Pressure systolic', 'NBPsys', 'Numeric', 'Metavision', 'Vital Signs', 'mmHg', 'metavision'),
    (220180, 'Non Invasive Blood Pressure diastolic', 'NBPdias', 'Numeric', 'Metavision', 'Vital Signs', 'mmHg', 'metavision'),
    (220181, 'Non Invasive Blood Pressure mean', 'NBPmean', 'Numeric', 'Metavision', 'Vital Signs', 'mmHg', 'metavision'),

    (220210, 'Respiratory Rate', 'RR', 'Numeric', 'Metavision', 'Vital Signs', 'insp/min', 'metavision'),
    (618, 'Respiratory Rate', 'RR', 'Numeric', 'CareVue', 'Vital Signs', '/min', 'carevue'),

    (223761, 'Temperature Celsius', 'Temp', 'Numeric', 'Metavision', 'Vital Signs', 'C', 'metavision'),
    (678, 'Temperature C', 'Temp', 'Numeric', 'CareVue', 'Vital Signs', 'C', 'carevue'),

    (220277, 'O2 saturation pulseoxymetry', 'SpO2', 'Numeric', 'Metavision', 'Vital Signs', '%', 'metavision'),
    (646, 'SpO2', 'SpO2', 'Numeric', 'CareVue', 'Vital Signs', '%', 'carevue'),

    # Glasgow Coma Scale
    (198, 'GCS Total', 'GCS', 'Numeric', 'CareVue', 'Scores', 'None', 'carevue'),
    (226755, 'GCS Total', 'GCS', 'Numeric', 'Metavision', 'Scores', 'None', 'metavision'),

    # Urine Output
    (40055, 'Urine Out Foley', 'UO', 'Numeric', 'CareVue', 'Outputs', 'ml', 'carevue'),
    (40069, 'Urine Out Void', 'UO', 'Numeric', 'CareVue', 'Outputs', 'ml', 'carevue'),
    (226559, 'Foley', 'UO', 'Numeric', 'Metavision', 'Outputs', 'mL', 'metavision'),
]

D_ITEMS = pd.DataFrame(vital_items, columns=[
    'ITEMID', 'LABEL', 'ABBREVIATION', 'LINKSTO',
    'CATEGORY', 'PARAM_TYPE', 'UNITNAME', 'DBSOURCE'
])

print(f"D_ITEMS table created: {len(D_ITEMS)} items")

CREATING TABLE 4/12: D_ITEMS (Dictionary)
D_ITEMS table created: 19 items


In [ ]:
print("CREATING TABLE 5/12: CHARTEVENTS (This may take a moment...)")

def generate_vital_timeseries(icustay_id, intime, outtime, vital_itemids, base_value, std_dev):
    """Generate realistic time-series for a vital sign"""
    duration_hours = (outtime - intime).total_seconds() / 3600
    if duration_hours <= 0:
      return pd.DataFrame()
    num_measurements = int(duration_hours * 2)  # ~2 measurements per hour

    if num_measurements == 0:
        return pd.DataFrame()

    # Generate timestamps
    timestamps = [intime + timedelta(hours=i/2) for i in range(num_measurements)]

    # Generate values with random walk
    values = [base_value]
    for _ in range(num_measurements - 1):
        change = np.random.normal(0, std_dev)
        values.append(max(0, values[-1] + change))

    # Create dataframe
    return pd.DataFrame({
        'ICUSTAY_ID': icustay_id,
        'CHARTTIME': timestamps,
        'ITEMID': np.random.choice(vital_itemids, num_measurements),
        'VALUE': [f"{v:.1f}" for v in values],
        'VALUENUM': values
    })

# Generate CHARTEVENTS for each ICU stay
all_chartevents = []

vital_configs = {
    'heart_rate': ([211, 220045], 80, 10),
    'map': ([220052, 220181], 80, 8),
    'respiratory_rate': ([220210, 618], 18, 3),
    'temperature': ([223761, 678], 37, 0.5),
    'spo2': ([220277, 646], 97, 2),
    'gcs': ([198, 226755], 14, 1.5)
}

print("Generating time-series vital signs for each ICU stay...")
for idx, row in ICUSTAYS.iterrows():
    if idx % 10 == 0:
        print(f"  Processing ICU stay {idx+1}/{len(ICUSTAYS)}")

    for vital_name, (itemids, base_value, std_dev) in vital_configs.items():
        vital_ts = generate_vital_timeseries(
            row['ICUSTAY_ID'],
            row['INTIME'],
            row['OUTTIME'],
            itemids,
            base_value,
            std_dev
        )
        all_chartevents.append(vital_ts)

CHARTEVENTS = pd.concat(all_chartevents, ignore_index=True)
CHARTEVENTS['ROW_ID'] = range(1, len(CHARTEVENTS) + 1)
CHARTEVENTS['SUBJECT_ID'] = CHARTEVENTS['ICUSTAY_ID'].map(
    dict(zip(ICUSTAYS['ICUSTAY_ID'], ICUSTAYS['SUBJECT_ID']))
)
CHARTEVENTS['HADM_ID'] = CHARTEVENTS['ICUSTAY_ID'].map(
    dict(zip(ICUSTAYS['ICUSTAY_ID'], ICUSTAYS['HADM_ID']))
)

# Reorder columns
CHARTEVENTS = CHARTEVENTS[[
    'ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID',
    'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM'
]]

print(f"\nCHARTEVENTS table created: {len(CHARTEVENTS):,} measurements")


CREATING TABLE 5/12: CHARTEVENTS (This may take a moment...)
Generating time-series vital signs for each ICU stay...
  Processing ICU stay 2201/234
  Processing ICU stay 1251/234
  Processing ICU stay 731/234
  Processing ICU stay 4621/234
  Processing ICU stay 5991/234
  Processing ICU stay 1911/234
  Processing ICU stay 2621/234
  Processing ICU stay 3501/234
  Processing ICU stay 2411/234
  Processing ICU stay 3421/234
  Processing ICU stay 301/234
  Processing ICU stay 2441/234
  Processing ICU stay 1891/234
  Processing ICU stay 6061/234
  Processing ICU stay 3491/234
  Processing ICU stay 3451/234
  Processing ICU stay 6721/234
  Processing ICU stay 5701/234
  Processing ICU stay 6281/234
  Processing ICU stay 2341/234
  Processing ICU stay 3471/234
  Processing ICU stay 4651/234
  Processing ICU stay 3511/234

CHARTEVENTS table created: 138,192 measurements


In [ ]:
print("CREATING TABLE 6/12: LABEVENTS")

print(observations_raw.columns)

# Filter observations for lab results
lab_obs = observations_raw[observations_raw['CATEGORY'] == 'laboratory'].copy()

lab_obs['DATE'] = pd.to_datetime(lab_obs['DATE']).dt.tz_localize(None)
encounters_raw['START'] = pd.to_datetime(encounters_raw['START']).dt.tz_localize(None)
encounters_raw['STOP'] = pd.to_datetime(encounters_raw['STOP']).dt.tz_localize(None)

# Merge lab_obs with encounters by patient
lab_obs_with_enc = lab_obs.merge(encounters_raw, on='PATIENT', how='left')

lab_obs_with_enc['CODE'] = lab_obs_with_enc['CODE_x']
lab_obs_with_enc = lab_obs_with_enc.drop(columns=['CODE_x', 'CODE_y'], errors='ignore')

print("Rows after merge:", len(lab_obs_with_enc))
print("Rows with DATE >= START:", (lab_obs_with_enc['DATE'] >= lab_obs_with_enc['START']).sum())
print("Rows with DATE <= STOP:", (lab_obs_with_enc['DATE'] <= lab_obs_with_enc['STOP']).sum())

lab_obs_with_enc = lab_obs_with_enc[
    (lab_obs_with_enc['DATE'] >= lab_obs_with_enc['START']) &
    (lab_obs_with_enc['DATE'] <= lab_obs_with_enc['STOP'])
]

# Assign pseudo HADM_ID from encounter Id
lab_obs_with_enc['HADM_ID'] = lab_obs_with_enc['Id']
lab_obs_with_enc['SUBJECT_ID'] = lab_obs_with_enc['PATIENT'].map(subject_id_map)
unique_lab_codes = lab_obs_with_enc['CODE'].unique()
lab_itemid_map = dict(zip(unique_lab_codes, range(50001, 50001 + len(unique_lab_codes))))

LABEVENTS = pd.DataFrame({
    'ROW_ID': range(1, len(lab_obs_with_enc) + 1),
    'SUBJECT_ID': lab_obs_with_enc['SUBJECT_ID'],
    'HADM_ID': lab_obs_with_enc['HADM_ID'],
    'ITEMID': lab_obs_with_enc['CODE'].map(lab_itemid_map),
    'CHARTTIME': lab_obs_with_enc['DATE'],
    'VALUE': lab_obs_with_enc['VALUE'].astype(str),
    'VALUENUM': pd.to_numeric(lab_obs_with_enc['VALUE'], errors='coerce'),
    'VALUEUOM': lab_obs_with_enc['UNITS'],
    'FLAG': None
})

print(f"LABEVENTS table created: {len(LABEVENTS):,} lab results")


CREATING TABLE 6/12: LABEVENTS
Index(['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION',
       'VALUE', 'UNITS', 'TYPE'],
      dtype='object')
Rows after merge: 14745267
Rows with DATE >= START: 6850682
Rows with DATE <= STOP: 7954467
LABEVENTS table created: 59,882 lab results


In [ ]:
print("CREATING TABLE 7/12: DIAGNOSES_ICD")

def extract_fluid(description):
    description = description.lower()
    if 'urine' in description:
        return 'Urine'
    elif 'blood' in description or 'serum' in description or 'plasma' in description:
        return 'Blood'
    elif 'csf' in description or 'cerebrospinal' in description:
        return 'CSF'
    elif 'stool' in description or 'feces' in description:
        return 'Stool'
    elif 'sputum' in description or 'respiratory' in description:
        return 'Sputum'
    else:
        return 'Other'

unique_codes = lab_obs['CODE'].unique()
lab_itemid_map = dict(zip(unique_codes, range(50001, 50001 + len(unique_codes))))

lab_obs_indexed = lab_obs.set_index('CODE')


# Create D_LABITEMS dictionary
D_LABITEMS = pd.DataFrame({
    'ROW_ID': range(1, len(unique_codes) + 1),
    'ITEMID': [lab_itemid_map[code] for code in unique_codes],
    'LABEL': [lab_obs.loc[lab_obs['CODE'] == code, 'DESCRIPTION'].iloc[0] for code in unique_codes],
    'FLUID': [extract_fluid(lab_obs.loc[lab_obs['CODE'] == code, 'DESCRIPTION'].iloc[0]) for code in unique_codes],
    'CATEGORY': [lab_obs.loc[lab_obs['CODE'] == code, 'CATEGORY'].iloc[0] for code in unique_codes],
    'LOINC_CODE': unique_codes
})

print(f"D_LABITEMS table created: {len(D_LABITEMS)} lab item definitions")


CREATING TABLE 7/12: DIAGNOSES_ICD
D_LABITEMS table created: 153 lab item definitions


In [ ]:
print("CREATING TABLE 8/12: NOTEEVENTS")

note_templates = {
    'Discharge summary': [
        "Patient admitted with {diagnosis}. Clinical course notable for {course}. Discharged in {condition} condition.",
        "This {age} year old {gender} presented with {diagnosis}. Hospital course complicated by {complication}. Patient {outcome}.",
    ],
    'Nursing': [
        "Patient stable. Vital signs within normal limits. {vital_note}. Continue current management.",
        "Overnight shift: Patient {status}. {intervention} performed. No acute issues.",
    ],
    'Physician': [
        "Assessment and Plan: {diagnosis} - {plan}. Will continue to monitor closely.",
        "Progress Note: Patient showing {improvement}. Current medications adjusted. Follow-up labs pending.",
    ]
}

diagnoses_list = ['respiratory failure', 'sepsis', 'cardiac arrest', 'pneumonia', 'CHF exacerbation']
courses = ['ICU admission', 'mechanical ventilation', 'vasopressor support', 'intensive monitoring']
outcomes = ['improved', 'stabilized', 'requires continued care', 'transferred to floor']

notes_data = []

for idx, row in ADMISSIONS.iterrows():
    num_notes = np.random.randint(2, 6)

    for _ in range(num_notes):
        category = np.random.choice(list(note_templates.keys()))
        template = np.random.choice(note_templates[category])

        # Fill template
        age = np.random.randint(40, 85)
        gender = 'male' if np.random.random() > 0.5 else 'female'

        note_text = template.format(
            diagnosis=np.random.choice(diagnoses_list),
            age=age,
            gender=gender,
            course=np.random.choice(courses),
            condition=np.random.choice(['stable', 'improved', 'fair']),
            complication=np.random.choice(courses),
            outcome=np.random.choice(outcomes),
            status=np.random.choice(['resting comfortably', 'alert and oriented', 'stable']),
            intervention=np.random.choice(['medication adjustment', 'wound care', 'lab draw']),
            vital_note='HR 75-85, BP 120/80, RR 16-18',
            improvement=np.random.choice(['improvement', 'clinical progress', 'stability']),
            plan='continue current medications, monitor vitals'
        )

        chart_date = row['ADMITTIME'] + timedelta(days=np.random.randint(0, 5))

        notes_data.append({
            'SUBJECT_ID': row['SUBJECT_ID'],
            'HADM_ID': row['HADM_ID'],
            'CHARTDATE': chart_date.date(),
            'CHARTTIME': chart_date,
            'CATEGORY': category,
            'DESCRIPTION': category,
            'TEXT': note_text
        })

NOTEEVENTS = pd.DataFrame(notes_data)
NOTEEVENTS['ROW_ID'] = range(1, len(NOTEEVENTS) + 1)
NOTEEVENTS = NOTEEVENTS[['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'CHARTDATE',
                         'CHARTTIME', 'CATEGORY', 'DESCRIPTION', 'TEXT']]

print(f"NOTEEVENTS table created: {len(NOTEEVENTS)} clinical notes")

print("\nSample note:")
print(NOTEEVENTS.iloc[0]['TEXT'])


CREATING TABLE 8/12: NOTEEVENTS
NOTEEVENTS table created: 2047 clinical notes

Sample note:
Progress Note: Patient showing stability. Current medications adjusted. Follow-up labs pending.


In [ ]:
print("CREATING TABLE 9/12: DIAGNOSES_ICD")

print(encounters_raw.columns)
print(conditions_raw.columns)

# Get conditions that occurred during admissions
diagnoses_data = conditions_raw.merge(
    encounters_raw[['Id']],
    left_on='ENCOUNTER',
    right_on='Id',
    how='inner'
)

# Filter for admission encounters
diagnoses_data = diagnoses_data[
    diagnoses_data['ENCOUNTER'].isin(inpatient['Id'])
]

DIAGNOSES_ICD = pd.DataFrame({
    'ROW_ID': range(1, len(diagnoses_data) + 1),
    'SUBJECT_ID': diagnoses_data['PATIENT'].map(subject_id_map),
    'HADM_ID': diagnoses_data['ENCOUNTER'].map(hadm_id_map),
    'SEQ_NUM': diagnoses_data.groupby('ENCOUNTER').cumcount() + 1,
    'ICD9_CODE': diagnoses_data['CODE'],
    'ICD10_CODE': diagnoses_data['CODE']  # Synthea uses SNOMED, treating as ICD10
})

print(f"DIAGNOSES_ICD table created: {len(DIAGNOSES_ICD)} diagnoses")
DIAGNOSES_ICD.head()


CREATING TABLE 9/12: DIAGNOSES_ICD
Index(['Id', 'START', 'STOP', 'PATIENT', 'ORGANIZATION', 'PROVIDER', 'PAYER',
       'ENCOUNTERCLASS', 'CODE', 'DESCRIPTION', 'BASE_ENCOUNTER_COST',
       'TOTAL_CLAIM_COST', 'PAYER_COVERAGE', 'REASONCODE',
       'REASONDESCRIPTION'],
      dtype='object')
Index(['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'SYSTEM', 'CODE',
       'DESCRIPTION'],
      dtype='object')
DIAGNOSES_ICD table created: 443 diagnoses


,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE,ICD10_CODE
5,1,10001,100001,1,428251008,428251008
73,2,10003,100003,1,384709000,384709000
74,3,10003,100003,2,70704007,70704007
87,4,10004,100004,1,384709000,384709000
88,5,10004,100004,2,44465007,44465007


In [ ]:
print("CREATING TABLE 10/12: D_ICD_DIAGNOSES dictionary")

# Create D_ICD_DIAGNOSES dictionary
unique_diagnoses = diagnoses_data.drop_duplicates('CODE')
D_ICD_DIAGNOSES = pd.DataFrame({
    'ROW_ID': range(1, len(unique_diagnoses) + 1),
    'ICD9_CODE': unique_diagnoses['CODE'].values,
    'SHORT_TITLE': unique_diagnoses['DESCRIPTION'].str[:50].values,
    'LONG_TITLE': unique_diagnoses['DESCRIPTION'].values
})

print(f"D_ICD_DIAGNOSES table created: {len(D_ICD_DIAGNOSES)} diagnosis definitions")

CREATING TABLE 10/12: D_ICD_DIAGNOSES dictionary
D_ICD_DIAGNOSES table created: 76 diagnosis definitions


In [ ]:
print("CREATING TABLE 11/12: DRGCODES")

drg_mapping = {
    'respiratory': (189, 'PULMONARY EDEMA & RESPIRATORY FAILURE', 3, 3),
    'cardiac': (291, 'HEART FAILURE & SHOCK W MCC', 4, 4),
    'sepsis': (871, 'SEPTICEMIA OR SEVERE SEPSIS W/O MV 96+ HOURS W MCC', 4, 4),
    'pneumonia': (193, 'SIMPLE PNEUMONIA & PLEURISY W MCC', 3, 2),
    'default': (640, 'NUTRITIONAL & MISC METABOLIC DISORDERS W MCC', 2, 1)
}

drg_data = []
for idx, row in ADMISSIONS.iterrows():
    # Determine DRG based on diagnosis
    diagnosis = str(row['DIAGNOSIS']).lower()
    drg_type = 'default'
    for key in drg_mapping.keys():
        if key in diagnosis:
            drg_type = key
            break

    drg_code, description, severity, mortality = drg_mapping[drg_type]

    drg_data.append({
        'SUBJECT_ID': row['SUBJECT_ID'],
        'HADM_ID': row['HADM_ID'],
        'DRG_TYPE': 'MS',
        'DRG_CODE': str(drg_code),
        'DESCRIPTION': description,
        'DRG_SEVERITY': severity,
        'DRG_MORTALITY': mortality
    })

DRGCODES = pd.DataFrame(drg_data)
DRGCODES['ROW_ID'] = range(1, len(DRGCODES) + 1)

print(f"DRGCODES table created: {len(DRGCODES)} DRG assignments")


CREATING TABLE 11/12: DRGCODES
DRGCODES table created: 585 DRG assignments


In [ ]:
print("CREATING TABLE 12/12: PROCEDUREEVENTS_MV")

# Generate ventilation events for ~30% of ICU stays
vent_icustays = ICUSTAYS.sample(frac=0.3, random_state=42)

procedure_data = []

for idx, row in vent_icustays.iterrows():
    # Ventilation start and stop
    start_time = row['INTIME'] + timedelta(hours=np.random.randint(0, 12))
    duration_hours = np.random.randint(12, 72)
    end_time = start_time + timedelta(hours=duration_hours)

    procedure_data.append({
        'SUBJECT_ID': row['SUBJECT_ID'],
        'HADM_ID': row['HADM_ID'],
        'ICUSTAY_ID': row['ICUSTAY_ID'],
        'STARTTIME': start_time,
        'ENDTIME': end_time,
        'ITEMID': 225792,  # Admission Weight itemid, using as proxy for ventilation
        'VALUE': 'Mechanical Ventilation',
        'VALUEUOM': None,
        'LOCATION': 'ICU',
        'LOCATIONCATEGORY': 'ICU',
        'ORDERID': np.random.randint(1000000, 9999999)
    })

PROCEDUREEVENTS_MV = pd.DataFrame(procedure_data)
PROCEDUREEVENTS_MV['ROW_ID'] = range(1, len(PROCEDUREEVENTS_MV) + 1)

print(f"PROCEDUREEVENTS_MV table created: {len(PROCEDUREEVENTS_MV)} procedures")
print(f"ICU stays with ventilation: {len(vent_icustays)} ({len(vent_icustays)/len(ICUSTAYS)*100:.1f}%)")


CREATING TABLE 12/12: PROCEDUREEVENTS_MV
PROCEDUREEVENTS_MV table created: 70 procedures
ICU stays with ventilation: 70 (29.9%)


In [ ]:
# Save all to csv
PATIENTS.to_csv("PATIENTS.csv", index=False)
ADMISSIONS.to_csv("ADMISSIONS.csv", index=False)
ICUSTAYS.to_csv("ICUSTAYS.csv", index=False)
CHARTEVENTS.to_csv("CHARTEVENTS.csv", index=False)
LABEVENTS.to_csv("LABEVENTS.csv", index=False)
D_LABITEMS.to_csv("D_LABITEMS.csv", index=False)
D_ITEMS.to_csv("D_ITEMS.csv", index=False)
DIAGNOSES_ICD.to_csv("DIAGNOSES_ICD.csv", index=False)
NOTEEVENTS.to_csv("NOTEEVENTS.csv", index=False)
D_ICD_DIAGNOSES.to_csv("D_ICD_DIAGNOSES.csv", index=False)
DRGCODES.to_csv("DRGCODES.csv", index=False)
PROCEDUREEVENTS_MV.to_csv("PROCEDUREEVENTS_MV.csv", index=False)


In [ ]:
import os

folder_name = 'mimic_data'
os.makedirs(folder_name, exist_ok = True)

tables = {
    "ADMISSIONS": ADMISSIONS,
    "ICUSTAYS": ICUSTAYS,
    "CHARTEVENTS": CHARTEVENTS,
    "DIAGNOSES_ICD": DIAGNOSES_ICD,
    "DRGCODES": DRGCODES,
    "LABEVENTS": LABEVENTS,
    "NOTEEVENTS": NOTEEVENTS,
    "PATIENTS": PATIENTS,
    "PROCEDUREEVENTS_MV": PROCEDUREEVENTS_MV
}

for name, df in tables.items():
  path = os.path.join(folder_name, f"{name}.csv")
  df.to_csv(path, index=False)

!zip -r mimic_data.zip mimic_data/

from google.colab import files
files.download('mimic_data.zip')



  adding: mimic_data/ (stored 0%)
  adding: mimic_data/NOTEEVENTS.csv (deflated 93%)
  adding: mimic_data/PROCEDUREEVENTS_MV.csv (deflated 73%)
  adding: mimic_data/LABEVENTS.csv (deflated 88%)
  adding: mimic_data/CHARTEVENTS.csv (deflated 76%)
  adding: mimic_data/DIAGNOSES_ICD.csv (deflated 80%)
  adding: mimic_data/ADMISSIONS.csv (deflated 86%)
  adding: mimic_data/ICUSTAYS.csv (deflated 71%)
  adding: mimic_data/PATIENTS.csv (deflated 68%)
  adding: mimic_data/DRGCODES.csv (deflated 92%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

old (don't run)

In [ ]:
# Filter for inpatient/emergency encounters
inpatient_encounters = encounters[
    encounters['ENCOUNTERCLASS'].isin(['inpatient', 'emergency', 'urgent'])
].copy()

# Create patient ID mapping
patient_id_map = dict(zip(patients['Id'], range(10001, 10001 + len(patients))))

# Transform to MIMIC-III ADMISSIONS
mimic_admissions = pd.DataFrame({
    'ROW_ID': range(1, len(inpatient_encounters) + 1),
    'SUBJECT_ID': inpatient_encounters['PATIENT'].map(patient_id_map),
    'HADM_ID': range(100001, 100001 + len(inpatient_encounters)),
    'ADMITTIME': pd.to_datetime(inpatient_encounters['START']),
    'DISCHTIME': pd.to_datetime(inpatient_encounters['STOP']),
    'DEATHTIME': pd.to_datetime(inpatient_encounters['STOP']).where(
        inpatient_encounters['REASONDESCRIPTION'].str.contains('death|died', case=False, na=False)
    ),
    'ADMISSION_TYPE': inpatient_encounters['ENCOUNTERCLASS'].str.upper(),
    'ADMISSION_LOCATION': 'EMERGENCY ROOM ADMIT',  # Simplified
    'DISCHARGE_LOCATION': 'HOME',  # Simplified
    'ETHNICITY': 'WHITE',  # Simplified - Synthea has this in patients table
    'DIAGNOSIS': inpatient_encounters['REASONDESCRIPTION'],
    'HOSPITAL_EXPIRE_FLAG': inpatient_encounters['REASONDESCRIPTION'].str.contains(
        'death|died', case=False, na=False
    ).astype(int)
})

print("MIMIC-III ADMISSIONS Table:")
print(mimic_admissions.head())
print(f"\nShape: {mimic_admissions.shape}")

In [ ]:
# Create HADM_ID mapping from encounters
encounter_hadm_map = dict(zip(
    inpatient_encounters['Id'],
    range(100001, 100001 + len(inpatient_encounters))
))

# Filter conditions that occurred during inpatient encounters
diagnoses = conditions.merge(
    encounters[['Id', 'PATIENT', 'ENCOUNTERCLASS']],
    left_on='ENCOUNTER',
    right_on='Id',
    how='inner'
)
diagnoses = diagnoses[diagnoses['ENCOUNTERCLASS'].isin(['inpatient', 'emergency', 'urgent'])]

# Transform to MIMIC-III DIAGNOSES_ICD
mimic_diagnoses = pd.DataFrame({
    'ROW_ID': range(1, len(diagnoses) + 1),
    'SUBJECT_ID': diagnoses['PATIENT'].map(patient_id_map),
    'HADM_ID': diagnoses['ENCOUNTER'].map(encounter_hadm_map),
    'SEQ_NUM': diagnoses.groupby('ENCOUNTER').cumcount() + 1,
    'ICD9_CODE': diagnoses['CODE'],
    'DESCRIPTION': diagnoses['DESCRIPTION']
})

print("MIMIC-III DIAGNOSES_ICD Table:")
print(mimic_diagnoses.head())
print(f"\nShape: {mimic_diagnoses.shape}")

In [ ]:
# Filter for lab observations
lab_obs = observations[observations['TYPE'] == 'laboratory'].copy()

# Create HADM_ID from encounter
lab_obs_with_encounter = lab_obs.merge(
    encounters[['Id', 'PATIENT']],
    left_on='ENCOUNTER',
    right_on='Id',
    how='left'
)

# Transform to MIMIC-III LABEVENTS
mimic_labevents = pd.DataFrame({
    'ROW_ID': range(1, len(lab_obs_with_encounter) + 1),
    'SUBJECT_ID': lab_obs_with_encounter['PATIENT'].map(patient_id_map),
    'HADM_ID': lab_obs_with_encounter['ENCOUNTER'].map(encounter_hadm_map),
    'ITEMID': lab_obs_with_encounter['CODE'].astype('category').cat.codes + 50000,
    'CHARTTIME': pd.to_datetime(lab_obs_with_encounter['DATE']),
    'VALUE': lab_obs_with_encounter['VALUE'],
    'VALUENUM': pd.to_numeric(lab_obs_with_encounter['VALUE'], errors='coerce'),
    'VALUEUOM': lab_obs_with_encounter['UNITS'],
    'FLAG': None  # Would need logic to determine abnormal
})

print("MIMIC-III LABEVENTS Table:")
print(mimic_labevents.head(10))
print(f"\nShape: {mimic_labevents.shape}")
print(f"\nUnique lab tests: {mimic_labevents['ITEMID'].nunique()}")

In [ ]:
# Merge medications with encounters for HADM_ID
medications_with_encounter = medications.merge(
    encounters[['Id', 'PATIENT']],
    left_on='ENCOUNTER',
    right_on='Id',
    how='left'
)

# Transform to MIMIC-III PRESCRIPTIONS
mimic_prescriptions = pd.DataFrame({
    'ROW_ID': range(1, len(medications_with_encounter) + 1),
    'SUBJECT_ID': medications_with_encounter['PATIENT'].map(patient_id_map),
    'HADM_ID': medications_with_encounter['ENCOUNTER'].map(encounter_hadm_map),
    'STARTDATE': pd.to_datetime(medications_with_encounter['START']),
    'ENDDATE': pd.to_datetime(medications_with_encounter['STOP']),
    'DRUG': medications_with_encounter['DESCRIPTION'],
    'DRUG_NAME_GENERIC': medications_with_encounter['DESCRIPTION'],
    'FORMULARY_DRUG_CD': medications_with_encounter['CODE'],
    'NDC': medications_with_encounter['CODE'],  # Simplified
    'ROUTE': 'PO',  # Simplified - would need parsing
    'DOSE_VAL_RX': None,  # Not in Synthea by default
    'DOSE_UNIT_RX': None
})

print("MIMIC-III PRESCRIPTIONS Table:")
print(mimic_prescriptions.head())
print(f"\nShape: {mimic_prescriptions.shape}")
print(f"\nUnique medications: {mimic_prescriptions['DRUG'].nunique()}")

In [ ]:
# Create output directory
import os
os.makedirs('mimic_format', exist_ok=True)

# Save all MIMIC-III tables
mimic_patients.to_csv('mimic_format/PATIENTS.csv', index=False)
mimic_admissions.to_csv('mimic_format/ADMISSIONS.csv', index=False)
mimic_diagnoses.to_csv('mimic_format/DIAGNOSES_ICD.csv', index=False)
mimic_labevents.to_csv('mimic_format/LABEVENTS.csv', index=False)
mimic_prescriptions.to_csv('mimic_format/PRESCRIPTIONS.csv', index=False)

print("All MIMIC-III format files saved")
print("\nFiles created:")
!ls -lh mimic_format/

# Create a summary report
summary = f"""
MIMIC-III Format Generation Summary
=====================================
Patients: {len(mimic_patients):,}
Admissions: {len(mimic_admissions):,}
Diagnoses: {len(mimic_diagnoses):,}
Lab Events: {len(mimic_labevents):,}
Prescriptions: {len(mimic_prescriptions):,}

Mortality Rate: {mimic_patients['EXPIRE_FLAG'].mean():.1%}
Avg Age: {(pd.Timestamp.now() - mimic_patients['DOB']).dt.days.mean() / 365.25:.1f} years
"""

print(summary)

# Save summary
with open('mimic_format/GENERATION_SUMMARY.txt', 'w') as f:
    f.write(summary)

In [ ]:
# Zip all MIMIC files for easy download
!zip -r mimic_format.zip mimic_format/

# Download using Colab's file download
from google.colab import files
files.download('mimic_format.zip')

print("Download started")

In [ ]:
# Perform data quality checks
print("Data Quality Checks")
print("=" * 50)

# Check for missing critical values
print("\n1. Missing Values:")
print(f"Patients with no admissions: {len(mimic_patients) - mimic_admissions['SUBJECT_ID'].nunique()}")
print(f"Admissions with no diagnoses: {len(mimic_admissions) - mimic_diagnoses['HADM_ID'].nunique()}")

# Check date ranges
print("\n2. Date Ranges:")
print(f"DOB range: {mimic_patients['DOB'].min()} to {mimic_patients['DOB'].max()}")
print(f"Admission range: {mimic_admissions['ADMITTIME'].min()} to {mimic_admissions['ADMITTIME'].max()}")

# Check distributions
print("\n3. Gender Distribution:")
print(mimic_patients['GENDER'].value_counts())

print("\n4. Admission Type Distribution:")
print(mimic_admissions['ADMISSION_TYPE'].value_counts())

# Check for data integrity
print("\n5. Data Integrity:")
invalid_admissions = mimic_admissions[
    mimic_admissions['DISCHTIME'] < mimic_admissions['ADMITTIME']
]
print(f"Invalid admission dates (discharge before admit): {len(invalid_admissions)}")

print("\nQuality checks complete")